# TenderLens · S2 Data Cleaning
## Step 1 · Load the S1 table safely
We read every cell as text (no guessing) and work on a copy, so the original stays untouched.

In [91]:
import pandas as pd    # pandas helps us work with tables

In [92]:
# open the table we saved in S1
raw = pd.read_csv("../data/interim/raw_master.csv", dtype=str)

# make a copy to clean (the original "raw" stays safe)
df = raw.copy()

In [93]:
print(df.shape)    # how many rows and columns?
df.head(3)         # show the first 3 tenders

(39500, 26)


,file_name,file_type,tender_id,reference_no,department,title,work_description,category,product_category,tender_type,...,bid_opening_date,work_period_days,location,amendment_count,cancelled,has_award,number_of_bids,awarded_value,winning_supplier,published_year
0,2020_PWD_16002_215_0.csv,older,2020_PWD_16002_215,CE/Maint/A.D./404/2019/2,Public Works Roads Department,"Dhu/AD19-20/23 Improvement and upgradation of Key Infrastructure and connectivity K.G.T. road to Simlabari Rajohua Satra, Kaimari, Golakganj in Dhubri District","Dhu/AD19-20/23 Improvement and upgradation of Key Infrastructure and connectivity K.G.T. road to Simlabari Rajohua Satra, Kaimari, Golakganj in Dhubri District",Works,Civil Works,Open Tender,...,06-Feb-2020 02:00 PM,270.0,Dhubri,2,False,False,NaN,NaN,NaN,2020
1,2020_PWD_15864_63_0.csv,older,2020_PWD_15864_63,CE/DEV/TB/199/2019-20/01,Public Works Roads Department,SOPD_SIG_19_20_63,"SOPD_SIG_19_20_63 Improvement of road from Karichuk Tianiali to Dhemaji Engineering College under Signature Project of SOPD-G (State Priority Scheme) for the year 2019-20, Dhemaji LAC",Works,Civil Works,Open Tender,...,05-Feb-2020 02:10 PM,270.0,Dhemaji,1,False,False,NaN,NaN,NaN,2020
2,final_2022_GMC_28546_35.csv,newer,2022_GMC_28546_35,GER/CE/1806/2019/241 Dated-19/12/2022,Guwahati Municipal Corporation,Construction of Road and drain at Binoy Kumar Tamuli path at byelane no. 4,Improvement of Various works under Guwahati Municipal Corporation,Works,Civil Works – Roads,Open Tender,...,17-Jan-2023 02:30 PM,90.0,Guwahati,1,False,False,3.0,NaN,NaN,2023


### Insight · Step 1
- The table opened with 39,500 tenders and 26 columns, the same as at the end of S1, so nothing was lost.
- Each row is one tender, for example a road repair posted by the Public Works Roads Department.
- Right now every value is plain text, so Python cannot add money like "43,00,000" or compare dates yet.
- Only the newer files have a bidder list, so not every tender can tell us if it got one bid or many.
- Next I start a cleaning diary and check for tenders that appear twice.

## Step 2 · Start a cleaning diary
Every change we make gets one line in the diary.

### Make an empty dairy

In [94]:
diary = []    # an empty diary

In [95]:
diary.append({"step": "Loaded the table", "tenders_left": len(df)})

In [96]:
pd.DataFrame(diary)    # show the diary as a small table

,step,tenders_left
0,Loaded the table,39500


## Step 3 · Find tenders written twice (duplicates)
First we look at them. We decide what to remove afterwards.

In [97]:
df["tender_id"].nunique()    # count the different IDs

39411

In [98]:
repeats = df[df["tender_id"].duplicated(keep=False)]    # all rows whose ID appears more than once
len(repeats)                                             # how many such rows?

178

In [99]:
repeats[["file_name", "file_type", "tender_id", "number_of_bids"]].sort_values("tender_id").head(6)

,file_name,file_type,tender_id,number_of_bids
8610,final_2019_AHVD_15610_2.csv,newer,2019_AHVD_15610_2,4.0
36674,2019_AHVD_15610_2_0.csv,older,2019_AHVD_15610_2,NaN
17865,final_2019_BoTC_15648_1.csv,newer,2019_BoTC_15648_1,5.0
37055,2019_BoTC_15648_1_0.csv,older,2019_BoTC_15648_1,NaN
5313,final_2019_BoTC_15649_1.csv,newer,2019_BoTC_15649_1,4.0
29944,2019_BoTC_15649_1_0.csv,older,2019_BoTC_15649_1,NaN


In [100]:
repeats["file_type"].value_counts()    # how many older vs newer copies?

file_type
older    90
newer    88
Name: count, dtype: int64

## Step 4 · Remove duplicates
Keep the newer copy of each repeated tender (it has the bid count).

In [101]:
df = df.sort_values("file_type")    # "newer" comes before "older" (n comes before o in ABC)

In [102]:
df = df.drop_duplicates("tender_id", keep="first")    # keep the top copy, remove the rest
df = df.reset_index(drop=True)                        # renumber the rows 0, 1, 2, ...
len(df)                                               # how many tenders now?

39411

In [103]:
diary.append({"step": "Removed duplicate tenders (kept newer copy)", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411


### Insight · Duplicates
- 89 tenders were saved twice, which gave 178 rows for 89 jobs.
- In 88 of these, one copy was an older file without bidder details and the other was a newer "final_" file with the number of bids. I kept the newer copy because it has the information the model needs.
- One tender had two older copies, and I kept one of them.
- After removing the extra copies there are 39,411 unique tenders, so each job is counted only once.

## Step 5 · Tender value: text → number
Remove the commas, then turn the text into a real number.

In [104]:
df["estimated_value"].head(10)    # see what the values look like

0      43,00,000
1       5,00,000
2    7,49,99,615
3            NaN
4            NaN
5    3,19,95,687
6      13,98,000
7            NaN
8            NaN
9            0.0
Name: estimated_value, dtype: str

In [105]:
df["estimated_value"].isna().sum()    # how many tenders have no value?

3497

In [106]:
df["estimated_value"] = df["estimated_value"].str.replace(",", "")                 # remove the commas
df["estimated_value"] = pd.to_numeric(df["estimated_value"], errors="coerce")      # text → number

In [107]:
print(df["estimated_value"].isna().sum())    # empty values now (compare with Cell B)
print(df["estimated_value"].median())        # the middle tender value in ₹
print(df["estimated_value"].max())           # the biggest tender value in ₹

3497
9224349.5
20000000000.0


## Step 6 · Turn the other number columns into numbers
Same job as Step 5, done for 6 columns using a loop.

In [108]:
num_cols = ["emd_amount", "tender_fee", "awarded_value",   "number_of_bids", "work_period_days", "amendment_count"]

In [109]:
before = df[num_cols].isna().sum()    # empty values in each column, before

In [110]:
for col in num_cols:                                          # take each column, one by one
    df[col] = df[col].str.replace(",", "")                    # remove the commas
    df[col] = pd.to_numeric(df[col], errors="coerce")         # text → number

In [111]:
after = df[num_cols].isna().sum()                               # empty values, after
pd.DataFrame({"empty_before": before, "empty_after": after})    # show them side by side

,empty_before,empty_after
emd_amount,0,0
tender_fee,0,0
awarded_value,35801,35801
number_of_bids,32621,32621
work_period_days,0,1814
amendment_count,0,0


### Step 6b · Look at the work periods that were words

In [112]:
test = pd.to_numeric(raw["work_period_days"].str.replace(",", ""), errors="coerce")    # try turning into numbers

In [113]:
raw["work_period_days"][test.isna()].value_counts().head(15)    # the top 15 word values

work_period_days
NA     1822
Name: count, dtype: int64

In [114]:
diary.append({"step": "Turned 7 number columns from text into numbers", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411


## Step 7 · Dates: text → real dates
First we look at how the dates are written.

In [115]:
df[["published_date", "last_date", "bid_opening_date"]].head(5)    # first 5 tenders, only the date columns

,published_date,last_date,bid_opening_date
0,2023-01-07 09:00:00,17-Jan-2023 02:00 PM,17-Jan-2023 02:30 PM
1,2022-12-16 12:00:00,05-Jan-2023 04:00 PM,06-Jan-2023 11:30 AM
2,2022-04-06 18:00:00,12-Apr-2022 02:00 PM,12-Apr-2022 03:00 PM
3,2022-12-28 13:00:00,23-Jan-2023 02:00 PM,23-Jan-2023 04:00 PM
4,2022-08-18 15:00:00,02-Sep-2022 01:00 PM,02-Sep-2022 02:30 PM


In [116]:
date_cols = ["published_date", "last_date", "bid_opening_date"]    # our 3 date columns
before = df[date_cols].isna().sum()                                # empty dates, before

In [117]:
df["published_date"] = pd.to_datetime(df["published_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")

In [118]:
df["last_date"] = pd.to_datetime(df["last_date"], format="%d-%b-%Y %I:%M %p", errors="coerce")
df["bid_opening_date"] = pd.to_datetime(df["bid_opening_date"], format="%d-%b-%Y %I:%M %p", errors="coerce")

In [119]:
after = df[date_cols].isna().sum()                              # empty dates, after
pd.DataFrame({"empty_before": before, "empty_after": after})    # side by side

,empty_before,empty_after
published_date,0,0
last_date,0,0
bid_opening_date,0,0


In [120]:
diary.append({"step": "Turned 3 date columns from text into real dates", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411


## Step 8 · Bidding window (days companies get to bid)
bidding window = last_date − published_date

In [121]:
df["bid_window_days"] = (df["last_date"] - df["published_date"]).dt.days    # days between posting and deadline

In [122]:
# Look at the windows
print(df["bid_window_days"].min())       # the shortest window
print(df["bid_window_days"].median())    # the middle window
print(df["bid_window_days"].max())       # the longest window

-2167
14.0
1103


In [123]:
# Count the strange ones
print((df["bid_window_days"] < 0).sum())     # deadline BEFORE posting (impossible!)
print((df["bid_window_days"] > 90).sum())    # more than 3 months to bid (very unusual)

19
129


### Insight · Bidding window
- A typical tender gives companies about 14 days to prepare and send their bid.
- 19 tenders have a last date before the posting date (up to 2,167 days before). This is impossible and is most likely a wrong year typed in.
- 129 tenders give more than 90 days, the longest 1,103 days. A few may be real big projects, but most look like date mistakes.
- A short window could be one reason only one company bids, so these values need to be correct before I use them in the model.

## Step 8b · Look at the impossible windows

In [124]:
neg = df[df["bid_window_days"] < 0]                                          # only the impossible tenders
neg[["published_date", "last_date", "bid_window_days"]].head(8)              # see their dates

,published_date,last_date,bid_window_days
707,2021-12-23 18:00:00,2021-11-15 13:00:00,-39
5626,2023-04-25 14:45:00,2022-01-18 14:00:00,-463
7233,2023-05-16 11:39:00,2022-11-18 15:00:00,-179
8672,2022-10-13 14:00:00,2022-09-09 17:00:00,-34
10919,2022-06-23 14:42:00,2018-03-05 16:30:00,-1571
11413,2020-11-06 17:15:00,2020-10-22 13:00:00,-16
11440,2022-12-16 08:34:00,2017-12-26 14:00:00,-1816
16298,2019-10-21 16:15:00,2019-10-11 14:00:00,-11


In [125]:
long = df[df["bid_window_days"] > 90]                                        # only the very long windows
long[["published_date", "last_date", "bid_window_days"]].sort_values("bid_window_days").tail(8)    # the 8 longest

,published_date,last_date,bid_window_days
31175,2017-05-26 15:00:00,2018-01-08 15:00:00,227
24497,2021-02-23 13:00:00,2021-10-28 14:00:00,247
907,2020-03-07 11:00:00,2020-11-09 11:00:00,247
13059,2017-10-27 11:00:00,2018-08-21 15:00:00,298
2296,2021-02-22 18:40:00,2022-01-04 14:00:00,315
2356,2021-08-09 17:40:00,2022-06-23 17:00:00,317
204,2021-05-18 16:00:00,2022-04-18 14:00:00,334
16629,2019-02-08 14:00:00,2022-02-15 17:40:00,1103


### Step 8c · Fix and mark the bidding windows
Negative window → empty (unknown). Very long window → keep, but mark it.

In [126]:
import numpy as np    # numpy gives us np.nan = an empty value

In [127]:
# Make a label column
df["window_flag"] = "ok"                                              # start: every tender is "ok"
df.loc[df["bid_window_days"] < 0, "window_flag"] = "negative"         # mark the impossible ones
df.loc[df["bid_window_days"] > 90, "window_flag"] = "very long"       # mark the very long ones
df["window_flag"].value_counts()                                      # count each label

window_flag
ok           39263
very long      129
negative        19
Name: count, dtype: int64

In [128]:
# Make the negative windows empty
df.loc[df["bid_window_days"] < 0, "bid_window_days"] = np.nan    # impossible window → unknown
print(df["bid_window_days"].min())                                # shortest window now

2.0


In [129]:
diary.append({"step": "Bidding window: 19 negative → empty, 129 very long → marked", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411


## Step 9 · Missing and ₹0 tender values
₹0 means "not given", so it becomes empty. We mark missing values but don't fill them yet.

In [130]:
# How many say ₹0?
(df["estimated_value"] == 0).sum()    # tenders with value exactly ₹0

1885

In [131]:
df.loc[df["estimated_value"] == 0, "estimated_value"] = np.nan    # ₹0 → unknown

In [132]:
print(df["estimated_value"].isna().sum())               # total missing values
print(df["estimated_value"].isna().mean() * 100)        # as a percentage

5382
13.65608586435259


In [133]:
df["value_missing"] = df["estimated_value"].isna()    # True = value not given
df["value_missing"].value_counts()

value_missing
False    34029
True      5382
Name: count, dtype: int64

In [134]:
diary.append({"step": "Tender value: ₹0 → empty, added value_missing mark", "tenders_left": len(df)})
pd.DataFrame(diary)


,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411


### Insight · Missing tender values
- 5,382 of 39,411 tenders (13.7%, about 1 in 7) do not tell us the value of the job.
- 3,497 left the value blank and 1,885 wrote ₹0, which is not a real price. I treated both as unknown.
- I added a value_missing column, because a tender that hides its price might get fewer bidders.
- I did not fill these gaps yet. That will be done later using only the training tenders, so the model gets no hints from the future.

## Step 10 · Giant tender values
Look at tenders above ₹100 crore before deciding anything.

In [135]:
print((df["estimated_value"] > 1e9).sum())     # above ₹100 crore
print((df["estimated_value"] > 1e10).sum())    # above ₹1,000 crore

188
6


In [136]:
big = df[df["estimated_value"] > 1e9]                                                      # only the giant tenders
big[["department", "title", "estimated_value"]].sort_values("estimated_value").tail(10)    # the 10 biggest

,department,title,estimated_value
5657,Assam Power Distribution Company Ltd||CGM(PP and D) - APDCL,Appointment of AMISP for Smart Prepaid Metering in APDCL on DBFOOT basis under RDSS (PKG-5),7.681798e+09
6694,Public Works Roads Department-Externally Aided Project||Public Works Roads Department - AIIB Tenders,"Imp and upgdn of rd from Majuli to Lakhimpur, incl. constn. of two major bridges over Luit and Subansiri rivers on EPC mode",7.853640e+09
20840,Public Works Building and NH Department||Chief Engineer-Building - PWBNH,"Consultancy Services for Construction of proposed 2nd Medical College Centering MMCH Kalapahar campus along with 800 Bedded General Hospital at MMCH Campus in Guwahati, Assam",8.000000e+09
408,Public Works Building and NH Department||Chief Engineer-Building - PWBNH,PMC service for Construction of new medical college centering MMCH and Kalapahar campus along with 800 bedded General Hospital at MMCH Campus in Guwahati,9.542100e+09
7032,Public Works Building and NH Department||Chief Engineer-Building - PWBNH,"Consultancy Services forthe work Super Speciality wing at six Medical Colleges Silchar, Diphu, Lakhimpur, Tezpur, Jorhat and FAAMCH, Barpeta Infrastructure improvement in 6 (six) District Hospitals viz. Silchar, Udalguri, Hojai, Hailakandi, Darrang a",1.178000e+10
37152,Karbi Anglong Autonomous Council||PWD||Addl. Chief Engineer Diphu,Construction of raod with Mettalling and Black Topping from Era gaon to Tikoklangso towards Langchitung,1.491376e+10
9052,Assam Power Distribution Company Ltd||CGM(PP and D) - APDCL,Appointment of AMISP for Smart Prepaid Metering in APDCL on DBFOOT basis under RDSS (PKG-7),1.509679e+10
18736,Food Civil Supplies and Consumer Affairs Department||Directorate of Food Civil Supplies and Consumer Affairs,supply of sugar for distribution through Targeted Public Distribution System in Assam.,1.980396e+10
21043,Industries and Commerce Department||Assam Industrial Development Corporation Ltd - ICD,"Development of Twin Tower at Guwahati, Assam, India",2.000000e+10
14693,Industries and Commerce Department||Assam Industrial Development Corporation Ltd - ICD,"Development of Twin Tower at Guwahati, Assam, India",2.000000e+10


In [137]:
df["giant_value"] = df["estimated_value"] > 1e10    # True = above ₹1,000 crore
df["giant_value"].value_counts()

giant_value
False    39405
True         6
Name: count, dtype: int64

In [138]:
diary.append({"step": "Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411


### Insight · Giant tender values
- 188 tenders are worth more than ₹100 crore, and 6 are above ₹1,000 crore. The largest is ₹2,000 crore.
- When I read their titles, most were real large projects, like smart electricity meters for the whole state, a World Bank-aided road and a twin tower in Guwahati. So I kept them.
- A few look doubtful, like an ₹800 crore consultancy or a single local road at ₹1,491 crore, where the value may be the total project cost.
- I marked the 6 largest with giant_value and will use a log scale later so these values do not overpower the model.

## Step 11 · Award ratio = awarded value ÷ estimated value
Used only for the unusual-tender check later, never for prediction.

In [139]:
df["award_ratio"] = df["awarded_value"] / df["estimated_value"]    # final price ÷ expected price

In [140]:
print(df["award_ratio"].notna().sum())    # how many tenders have a ratio
print(df["award_ratio"].median())         # the middle ratio
print(df["award_ratio"].max())            # the biggest ratio

3130
0.9499965053693267
3658.0


In [141]:
print((df["award_ratio"] > 2).sum())      # paid more than DOUBLE the estimate
print((df["award_ratio"] < 0.5).sum())    # paid less than HALF the estimate

10
248


In [142]:
df[["title", "estimated_value", "awarded_value", "award_ratio"]].sort_values("award_ratio").tail(8)

,title,estimated_value,awarded_value,award_ratio
39403,SUPPLY OF TWO ROW MANUAL RICE TRANSPLANTER,NaN,NaN,NaN
39404,"Additional work for construction of Barama Cold Storage in Baksa District, BTC",21053228.0,NaN,NaN
39405,"Construction of RCC 3(three) storied Fire Station Building, Under Ground Reservoir (1.00 Lakh Ltr. Capacity), Boundary Wall and Security Fencing with Heavy Duty Iron Gate etc.at Mandia in Barpeta",18775258.0,NaN,NaN
39406,SENL/SOPD/2018-19/BAR-121,15500000.0,NaN,NaN
39407,"KAM/GRC/SOPD(G)/ 2020-21/52.Improvement of Purnajyoti Nagar Road with Guard Wall, Slab Culvert etc. at Zamartal, Mirza under SOPD (G) for the Year 2020-21.",NaN,NaN,NaN
39408,Construction of Dr. B. R. Ambedkar Bhawan at Gohpur Sub Division,17624000.0,NaN,NaN
39409,SE/SIL/SOPD/G/5,10200000.0,NaN,NaN
39410,"Works under Tejasvi Navadhitamastu Edu-Infra Fund (TNEIF) (Repair/ renovation works) at Silchar Medical College and Hospital, Silchar. Group-E Repair/ Renovation of Internal Roads",27148163.0,NaN,NaN


In [143]:
df.nlargest(8, "award_ratio")[["title", "estimated_value", "awarded_value", "award_ratio"]]

,title,estimated_value,awarded_value,award_ratio
5735,Techno-Economical Feasibility Study on Hydrokinetics Energy Generation,1000.0,3658000.00,3658.000000
3243,Valves,120.0,330988.82,2758.240167
8704,Preparation of Strategy Report cum Study of Cost Optimal Energy Mix Generation,1000.0,1746400.00,1746.400000
8202,SCI INTERIOR RETENDER,5000000.0,74561537.00,14.912307
307,Construction of Buildingless Sub Health Centre in Rangmandu SC in West Karbi Anglong District under PM ABHIM for the FY 2022 23 under NHM on Turnkey Basis,5203000.0,52000000.00,9.994234
5004,Supply of Non Dietary articles of Group F articles of Central Jail Dibrugarh,400000.0,1999320.00,4.998300
3988,ACCF/Medical Equipment/2021-22/38,5000000.0,19656000.00,3.931200
1246,Uniform Linen,1500000.0,4818261.00,3.212174


In [144]:
df.nsmallest(8, "award_ratio")[["title", "estimated_value", "awarded_value", "award_ratio"]]

,title,estimated_value,awarded_value,award_ratio
6310,"Appointment of Mandi Labour Contractors from the parties/Contractors with requisite work experience with strong financial background, for KMS 2022- 23",2802000.0,0.00,0.000000e+00
7479,"Appointment of Road Transport Contractors for Transportation of New Gunny Bales from Assam Co Operative Jute Mill Silghat to various PPCs under AFCSCL for KMS 2022- 23,, Silghat to Telem(Dhemaji Dist)",200000.0,0.00,0.000000e+00
8133,Biennial contract for supply of Medicines,1400000.0,0.00,0.000000e+00
5283,Essential Diagnostic Kit,9627000.0,1.00,1.038745e-07
5078,ReTender_Providing Onsite Training to Rural Masons through Recognition of Prior Learning under PMAY-G,31061923.0,53.90,1.735244e-06
4638,Sanitary Napkin,27151000.0,100.00,3.683106e-06
1127,Const of Khairabari Dev. Block Office Building under SOPD 2022-23,14563000.0,145.63,1.000000e-05
8817,Procurement of 300 Nos of Sugar Cane juicer Machine to be supplied in the office of the Block Dev. Offices under BTR area under SOPD 2022-23,29760000.0,297.60,1.000000e-05


### Step 11b · Mark unreliable award ratios
A ratio below 0.2 or above 5 is almost always a typing/unit mistake → make it empty and mark it.

In [145]:
bad_ratio = (df["award_ratio"] < 0.2) | (df["award_ratio"] > 5)    # too small OR too big
bad_ratio.sum()                                                    # how many?

197

In [146]:
df["award_flag"] = bad_ratio                       # True = prices don't match, check later
df.loc[bad_ratio, "award_ratio"] = np.nan          # unreliable ratio → unknown

In [147]:
print(df["award_ratio"].median())    # middle ratio
print(df["award_ratio"].min())       # smallest (should be 0.2 or more)
print(df["award_ratio"].max())       # biggest (should be 5 or less)

0.9500000418668951
0.20541786089861228
4.9983


In [148]:
diary.append({"step": "Award ratio: below 0.2 or above 5 → empty and marked", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411


### Insight · Award ratio
- For the 3,130 tenders where both prices are known, the middle tender was awarded at about 95% of its estimate. So the government usually paid about 5% less than expected.
- The extreme ratios were data-entry problems: placeholder estimates like ₹120 or ₹1,000, awards written in lakhs instead of rupees, and rate contracts with an award of ₹0.
- I treated ratios below 0.2 or above 5 as unreliable, made them empty and marked them (197 tenders).
- The award value is only known after bidding, so it will never be used to predict single bids. It is kept only for checking unusual tenders later.

## Step 12 · Department names
First we look at how many different names there are and how they are written.

In [149]:
df["department"].nunique()    # count of different names

454

In [150]:
df["department"].value_counts().head(15)    # top 15 names and how many tenders each posted

department
Public Works Roads Department                                                                                                            10445
Bodoland Territorial Council-PWD||Director Cum CHD - PWD BTC                                                                              3221
Department of Water Resources||Chief Engineer - DoWR                                                                                      2438
Public Works Building and NH Department||Chief Engineer-Building - PWBNH                                                                  1898
National Health Mission||State Programme Management Unit - NHM                                                                            1889
Guwahati Municipal Corporation                                                                                                            1034
Bodoland Territorial Council||PWD-BTC                                                                                              

###  Step 12b · Keep only the main department

In [151]:
#Cut the name at the first ||
df["dept_main"] = df["department"].str.split("||", regex=False).str[0].str.strip()

In [152]:
df["dept_main"].nunique()    # should be MUCH smaller than 454

106

In [153]:
#The 20 biggest main departments
df["dept_main"].value_counts().head(20)

dept_main
Public Works Roads Department                               11773
Bodoland Territorial Council-PWD                             3221
Public Works Building and NH Department                      3078
Bodoland Territorial Council                                 2499
Department of Water Resources                                2466
National Health Mission                                      2193
Industries and Commerce Department                           1288
Guwahati Municipal Corporation                               1034
Health and Family Welfare Department                          788
Assam Power Distribution Company Ltd                          765
Welfare of Plain Tribes and Backward Classes                  728
Home B                                                        656
Public Health Engineering Department                          600
Axom Sarba Siksha Abhijan Mission                             592
Irrigation Department                                         587


### Step 12c · Join the Bodoland twins

In [154]:
# Replace the twin name 
df["dept_main"] = df["dept_main"].replace("Bodoland Territorial Council-PWD", "Bodoland Territorial Council")

In [155]:
print(df["dept_main"].nunique())        # should be 105 now
df["dept_main"].value_counts().head(5)  # top 5 departments

105


dept_main
Public Works Roads Department              11773
Bodoland Territorial Council                5720
Public Works Building and NH Department     3078
Department of Water Resources               2466
National Health Mission                     2193
Name: count, dtype: int64

In [156]:
diary.append({"step": "Department: kept main name before || (454 → 105 names)", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411


### Insight · Departments
- Department names were written at many levels (department || chief engineer || zone || circle), which gave 454 different names.
- I kept only the main department and joined the two Bodoland Territorial Council names, which left 105 departments.
- Public Works Roads Department posted 11,773 tenders, about 30% of all tenders. Bodoland Territorial Council and Public Works Building and NH Department come next.
- A few departments post most of the tenders, so department is likely to be a strong clue for the model.

## Step 13 · The 2021 gap
Count tenders month by month to see how big the gap really is.

In [157]:
y2021 = df[df["published_date"].dt.year == 2021]              # only tenders posted in 2021
y2021["published_date"].dt.month.value_counts().sort_index()  # count per month (1 = Jan ... 12 = Dec)

published_date
1     1199
2     1523
3       32
4        2
5        5
6        9
7       13
8       14
9       78
10      12
11      31
12      98
Name: count, dtype: int64

In [158]:
# Tenders per month in 2022
y2022 = df[df["published_date"].dt.year == 2022]              # only tenders posted in 2022
y2022["published_date"].dt.month.value_counts().sort_index()  # count per month

published_date
1       52
2      143
3      289
4      362
5      461
6      462
7      433
8      393
9      508
10     541
11     855
12    1794
Name: count, dtype: int64

### Step 13b · Mark the gap period

In [159]:
# Mark tenders posted in the gap
df["gap_period"] = (df["published_date"] >= "2021-03-01") & (df["published_date"] < "2022-03-01")
df["gap_period"].sum()    # how many tenders fall inside the gap?

489

In [160]:
diary.append({"step": "Marked the Mar 2021–Feb 2022 gap (489 tenders)", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411
9,Marked the Mar 2021–Feb 2022 gap (489 tenders),39411


### Insight · The 2021 gap
- Tenders drop from about 1,500 in February 2021 to only 32 in March 2021, and stay very low for a whole year.
- There are only 489 tenders from March 2021 to February 2022, while January 2021 alone had 1,199.
- Elections and the COVID second wave may explain a few slow months, but a gap of a full financial year more likely means the data was not collected.
- I did not delete or fill anything. I marked these tenders with gap_period so later counts are not mistaken for a real slowdown.

## Step 14 · Re-tenders (same job posted again)
Mark a tender if its title says "retender", or if the same department posted the same title earlier.

In [161]:
# Make a simple version of the title
df["title_key"] = df["title"].str.lower().str.strip()    # small letters, no extra spaces

In [162]:
# Does the title say "retender"?
df["title_says_retender"] = df["title_key"].str.contains("retender|re-tender|re tender", na=False)
df["title_says_retender"].sum()

334

In [163]:
# Put tenders in date order
df = df.sort_values("published_date").reset_index(drop=True)    # oldest first

In [164]:
# Was the same job posted before?
df["seen_before"] = df.duplicated(["dept_main", "title_key"], keep="first")
df["seen_before"].sum()

5786

In [165]:
# Combine both ways
df["is_retender"] = df["title_says_retender"] | df["seen_before"]    # True if EITHER is true
df["is_retender"].value_counts()

is_retender
False    33319
True      6092
Name: count, dtype: int64

In [166]:
# Check with the Twin Tower example
twin = df[df["title_key"].str.contains("twin tower", na=False)]
twin[["published_date", "dept_main", "title", "is_retender"]]

,published_date,dept_main,title,is_retender
25169,2020-08-07 14:00:00,Industries and Commerce Department,Engagement of a Consultant for assisting the Government of Assam and the State Nodal Agency and collaborating with it in the development of Twin Tower Trade Centre in Guwahati,False
27104,2020-12-08 17:10:00,Industries and Commerce Department,"Development of Twin Tower at Guwahati, Assam, India",False
29809,2021-02-15 11:00:00,Industries and Commerce Department,"Development of Twin Tower at Guwahati, Assam, India",True


In [167]:
diary.append({"step": "Marked re-tenders (6,092: title says so or same job posted earlier)", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411
9,Marked the Mar 2021–Feb 2022 gap (489 tenders),39411


### Insight · Re-tenders
- 6,092 of 39,411 tenders (about 1 in 7) were posted again.
- 334 titles say "retender" directly, and 5,786 have the same department and title as an earlier tender.
- Example: the Guwahati Twin Tower was posted in December 2020 and again in February 2021.
- I only marked the later copy, because when a tender is posted we can only know about tenders that came before it.
- A tender that failed once may struggle again, so this could help predict single bids. Some repeats may be yearly orders, which I will check in EDA.

## Step 15 · The single-bid label
1 = exactly one bid, 0 = more than one bid, empty = unknown (never 0).

In [168]:
# Look at the bid counts first
df["number_of_bids"].value_counts().sort_index().head(10)    # how many tenders got 0, 1, 2, 3... bids

number_of_bids
1.0     2044
2.0     1212
3.0     1505
4.0      770
5.0      490
6.0      283
7.0      172
8.0      103
9.0       62
10.0      52
Name: count, dtype: int64

In [169]:
# Make the label
df["single_bid"] = np.nan                                  # start: everyone is "unknown"
df.loc[df["number_of_bids"] == 1, "single_bid"] = 1        # exactly one bid → 1
df.loc[df["number_of_bids"] > 1, "single_bid"] = 0         # more than one bid → 0

In [170]:
# Count the labels
df["single_bid"].value_counts(dropna=False)    # 1s, 0s and unknowns

single_bid
NaN    32621
0.0     4746
1.0     2044
Name: count, dtype: int64

In [171]:
# What percent are single-bid?
df["single_bid"].mean() * 100    # % of KNOWN tenders that got only one bid

30.103092783505154

In [172]:
diary.append({"step": "Made single_bid label: 2,044 ones, 4,746 zeros, rest unknown", "tenders_left": len(df)})
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411
9,Marked the Mar 2021–Feb 2022 gap (489 tenders),39411


### Insight · Single-bid label
- 6,790 of 39,411 tenders have a bidder list. 2,044 of them (30.1%) got only one bid, about 3 in every 10.
- One bid is the most common result, more common than two bids (1,212) or three bids (1,505).
- The other 32,621 tenders are kept as unknown, not 0, because their files have no bidder list.
- The classes are 30% single-bid and 70% multi-bid, so they are unbalanced. I will handle this later with class weights and threshold tuning.

## Step 16 · Save the clean table, the diary and the data dictionary

In [173]:
# Save the clean table
df.to_csv("../data/interim/tenders_clean.csv", index=False)    # save the clean table

In [174]:
# Last diary line, then save the diary
diary.append({"step": "Saved clean table to data/interim/tenders_clean.csv", "tenders_left": len(df)})
pd.DataFrame(diary).to_csv("../reports/cleaning_log.csv", index=False)

In [175]:
# Write the data dictionary
dictionary = [
    {"column": "dept_main", "meaning": "Main department (part before ||)", "use_for_prediction": "yes"},
    {"column": "bid_window_days", "meaning": "Days between posting and last date to bid", "use_for_prediction": "yes"},
    {"column": "window_flag", "meaning": "ok / negative (made empty) / very long (>90 days)", "use_for_prediction": "yes"},
    {"column": "value_missing", "meaning": "True if tender value was blank or ₹0", "use_for_prediction": "yes"},
    {"column": "giant_value", "meaning": "True if tender value above ₹1,000 crore", "use_for_prediction": "yes"},
    {"column": "gap_period", "meaning": "True if posted Mar 2021 – Feb 2022 (incomplete year)", "use_for_prediction": "yes"},
    {"column": "title_key", "meaning": "Title in small letters, no extra spaces", "use_for_prediction": "yes"},
    {"column": "is_retender", "meaning": "True if title says retender or same job posted earlier", "use_for_prediction": "yes"},
    {"column": "award_ratio", "meaning": "Awarded value ÷ estimated value (unreliable ones made empty)", "use_for_prediction": "NO - known only after bidding"},
    {"column": "award_flag", "meaning": "True if award ratio below 0.2 or above 5", "use_for_prediction": "NO - known only after bidding"},
    {"column": "single_bid", "meaning": "ANSWER: 1 = one bid, 0 = more than one, empty = unknown", "use_for_prediction": "this is the answer"},
]
pd.DataFrame(dictionary)

,column,meaning,use_for_prediction
0,dept_main,Main department (part before ||),yes
1,bid_window_days,Days between posting and last date to bid,yes
2,window_flag,ok / negative (made empty) / very long (>90 days),yes
3,value_missing,True if tender value was blank or ₹0,yes
4,giant_value,"True if tender value above ₹1,000 crore",yes
5,gap_period,True if posted Mar 2021 – Feb 2022 (incomplete year),yes
6,title_key,"Title in small letters, no extra spaces",yes
7,is_retender,True if title says retender or same job posted earlier,yes
8,award_ratio,Awarded value ÷ estimated value (unreliable ones made empty),NO - known only after bidding
9,award_flag,True if award ratio below 0.2 or above 5,NO - known only after bidding


In [176]:
# Save the dictionary
pd.DataFrame(dictionary).to_csv("../reports/data_dictionary.csv", index=False)

In [177]:
# Check that the clean table really saved
check = pd.read_csv("../data/interim/tenders_clean.csv")    # open the saved file again
check.shape                                                 # rows and columns

(39411, 39)

In [178]:
# show the final diary
pd.DataFrame(diary)    # the full story of our cleaning, step by step

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411
9,Marked the Mar 2021–Feb 2022 gap (489 tenders),39411


In [179]:
pd.set_option("display.max_colwidth", None)    # show full text, don't cut with "..."
pd.DataFrame(diary)

,step,tenders_left
0,Loaded the table,39500
1,Removed duplicate tenders (kept newer copy),39411
2,Turned 7 number columns from text into numbers,39411
3,Turned 3 date columns from text into real dates,39411
4,"Bidding window: 19 negative → empty, 129 very long → marked",39411
5,"Tender value: ₹0 → empty, added value_missing mark",39411
6,"Giant values: kept (mostly real projects), 6 above ₹1,000 cr marked",39411
7,Award ratio: below 0.2 or above 5 → empty and marked,39411
8,Department: kept main name before || (454 → 105 names),39411
9,Marked the Mar 2021–Feb 2022 gap (489 tenders),39411


## S2 Summary - Data Cleaning

### Starting point
- I started from raw_master.csv with 39,500 rows and 26 columns.

### What I cleaned
- Removed 89 duplicate tender IDs and kept the newer "final_" copy. This left 39,411 tenders.
- Changed 7 number columns from text to numbers. "NA" text in work_period_days became a proper empty value.
- Changed 3 date columns to real dates. None of them failed.
- Did not fill any missing values yet. I will do that in S4 using only training data, so no future information leaks in.

### New columns I made
- bid_window_days: days between publishing and the last bid date (median 14). 19 negative values were set to empty, and 129 windows over 90 days were kept.
- value_missing: the tender has no value or a value of 0. This covers 5,382 tenders (13.7%).
- giant_value: tenders over 1,000 crore. Only 6, and most look like real big projects, so I kept them.
- award_ratio and award_flag: winning price compared with the estimate. 197 odd cases were flagged. These are for analysis only, not for prediction.
- dept_main: the main department name. Bodoland Territorial Council-PWD was merged into Bodoland Territorial Council, giving 105 departments.
- gap_period: March 2021 to February 2022 has very few tenders (489), probably a data collection gap. I marked it but did not remove it.
- title_says_retender (334), seen_before (5,786) and is_retender (6,092) to spot tenders posted again.
- single_bid: 1 = one bid (2,044), 0 = more than one (4,746), empty = unknown (32,621).

### What I found
- The single-bid rate is 30.1% among tenders with a known bid count.
- Most tenders with bids got 1, 2 or 3 bids.
- The biggest departments are PWD Roads, Bodoland Territorial Council, PWD Building & NH, Water Resources and NHM.

### Files saved
- data/interim/tenders_clean.csv (39,411 rows, 39 columns)
- reports/cleaning_log.csv (a diary of 13 cleaning steps)
- reports/data_dictionary.csv (the 11 new columns and whether they can be used for prediction)

### Next steps (S3)
- Explore which kinds of tenders get only one bid: by year, department, bidding window, re-tender, value and month.
- Check the results again using SQL.